In [1]:
# PySpark Imports
import pyspark
from pyspark.sql import SparkSession

# ML Classifier Imports
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.sql.functions import mean, col
import time
import os
import sys

In [2]:
# Initialize Spark session
spark = SparkSession.builder.appName("ce53") \
    .master("local[*]") \
    .config("spark.driver.cores", "2") \
    .config("spark.driver.memory", "14g") \
    .config("spark.executor.memory", "14g") \
    .config("spark.executor.cores", "2") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "2") \
    .config("spark.dynamicAllocation.maxExecutors", "2") \
    .config("spark.executor.instances", "2") \
    .config("spark.kryoserializer.buffer.max", "2047m") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.executor.heartbeatInterval","11999s") \
    .config("spark.network.timeout","12000s") \
.getOrCreate()

24/04/08 21:03:54 WARN Utils: Your hostname, colin-MS-7977 resolves to a loopback address: 127.0.1.1; using 192.168.0.164 instead (on interface wlx3c52a1d3ccda)
24/04/08 21:03:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/08 21:03:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
parquet_files = ["Parquet/part-00000-1da06990-329c-4e38-913a-0f0aa39b388d-c000.snappy.parquet", "Parquet/part-00000-26e9208e-7819-451b-b23f-2e47f6d1e834-c000.snappy.parquet", 
                 "Parquet/part-00000-36240b61-b84f-4164-a873-d7973e652780-c000.snappy.parquet", "Parquet/part-00000-3f86626a-1225-47f9-a5a2-0170b737e404-c000.snappy.parquet",
                 "Parquet/part-00000-7c2e9adb-5430-4792-a42b-10ff5bbd46e8-c000.snappy.parquet", "Parquet/part-00000-b1a9fc13-8068-4a5d-91b2-871438709e81-c000.snappy.parquet",
                 "Parquet/part-00000-cbf26680-106d-40e7-8278-60520afdbb0e-c000.snappy.parquet", "Parquet/part-00000-df678a79-4a73-452b-8e72-d624b2732f17-c000.snappy.parquet"]

In [4]:
# Read the parquet files into a dataframe
df = spark.read.parquet(*parquet_files, inferSchema=True)

In [5]:
# Get unique labels and their counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the results
label_counts.show()

+--------------------+-------+
|        label_tactic|  count|
+--------------------+-------+
|   Credential Access|     31|
|     Defense Evasion|      1|
|           Discovery|   2086|
|        Exfiltration|      7|
|      Initial Access|      1|
|    Lateral Movement|      4|
|         Persistence|      1|
|Privilege Escalation|     13|
|      Reconnaissance|9278722|
|Resource Development|      3|
|                none|9281599|
+--------------------+-------+



In [6]:
start_time = time.time()

# List of labels to drop
labels_to_drop = ["Defense Evasion", "Exfiltration", "Initial Access", "Lateral Movement", "Persistence", "Privilege Escalation", "Resource Development", "Credential Access", "Reconnaissance"]

# Filter out the rows with labels to drop
df = df.filter(~col("label_tactic").isin(labels_to_drop))

# Get unique labels and their counts after filtering
filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the filtered results
filtered_label_counts.show()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

+------------+-------+
|label_tactic|  count|
+------------+-------+
|   Discovery|   2086|
|        none|9281599|
+------------+-------+

Execution time: 0.7962260246276855 seconds


In [7]:
start_time = time.time()



df = df.withColumn("datetime", col("datetime").cast("string"))

# Define columns to index
columns_to_index = ['service', 'conn_state', 'history', 'proto', 'dest_ip_zeek', 'community_id', 'uid', 'src_ip_zeek', 'label_tactic', 'datetime']

# Impute null values with 'null' string
for column in columns_to_index:
    df = df.fillna('null', subset=[column])

# Apply StringIndexer to each column
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").fit(df) for column in columns_to_index]

# Chain indexers together
pipeline = Pipeline(stages=indexers)

# Fit and transform the data
df_indexed = pipeline.fit(df).transform(df)

# Drop original columns
df_indexed = df_indexed.drop(*columns_to_index)

# Show the schema of the DataFrame
#df_indexed.show()



end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 46.92876601219177 seconds


In [8]:
# Split the data into training and test sets
start_time = time.time()

train_data, test_data = df_indexed.randomSplit([0.7, 0.3], seed=42)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.022047758102416992 seconds


In [9]:
from pyspark.ml.feature import Imputer


start_time = time.time()

# List of numeric column names
numeric_columns = ['resp_pkts', 'orig_ip_bytes', 'missed_bytes', 'duration', 'orig_pkts',
                   'resp_ip_bytes', 'dest_port_zeek', 'orig_bytes', 'resp_bytes',
                   'src_port_zeek', 'ts']


# Create an Imputer object
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

# Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data)

# Apply the imputer to the training data
train_data_imputed = imputer_model.transform(train_data)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()


# Apply the imputer to the test data
test_data_imputed = imputer_model.transform(test_data)

# Show updated DataFrames
#train_data_imputed.show()
#test_data_imputed.show()



end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/08 21:05:03 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


Execution time: 40.26534366607666 seconds
Execution time: 0.02279806137084961 seconds


In [10]:
from pyspark.ml.feature import VectorAssembler


start_time = time.time()



# List of columns to assemble
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed")]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

# Transform the training DataFrame
train_data_assembled = assembler.transform(train_data_imputed)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()
# Transform the test DataFrame
test_data_assembled = assembler.transform(test_data_imputed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



# Select only the features and label columns for both training and test sets
train_data_assembled = train_data_assembled.select("features", "label_tactic_indexed")
test_data_assembled = test_data_assembled.select("features", "label_tactic_indexed")

# Show the schema of the assembled training DataFrame
#train_data_assembled.printSchema()

# Show the schema of the assembled test DataFrame
#test_data_assembled.printSchema()





Execution time: 4.776498317718506 seconds
Execution time: 0.07246184349060059 seconds


In [11]:
from pyspark.ml.feature import StandardScaler


start_time = time.time()

# Standardize data on the training set
scaler = StandardScaler(inputCol="features", outputCol="features_normalized", withMean=True, withStd=True)
scaler_model = scaler.fit(train_data_assembled)
train_data_normalized = scaler_model.transform(train_data_assembled)
train_data_normalized = train_data_normalized.select("features_normalized", "label_tactic_indexed")


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/08 21:05:48 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:06:39 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


Execution time: 65.76197385787964 seconds


In [12]:
start_time = time.time()


# Apply the same transformation to the test set
test_data_normalized = scaler_model.transform(test_data_assembled)
test_data_normalized = test_data_normalized.select("features_normalized", "label_tactic_indexed")


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.03246450424194336 seconds


In [13]:
# Define the PCA model
pca = PCA(k=10, inputCol="features_normalized", outputCol="pca_features")

# Fit the PCA model on the normalized training set
start_time = time.time()
pca_model = pca.fit(train_data_normalized)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/08 21:07:04 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:07:37 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:08:04 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:08:44 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:09:05 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:09:32 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:09:46 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/04/08 21:10:09 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


Execution time: 210.8707971572876 seconds


24/04/08 21:10:13 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [14]:
# Apply PCA transformation to the training and test sets

start_time = time.time()
train_pca = pca_model.transform(train_data_normalized)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()
test_pca = pca_model.transform(test_data_normalized)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.03866076469421387 seconds
Execution time: 0.012810707092285156 seconds


In [15]:
# Drop the normalized column and rename the pca_features column
start_time = time.time()
train_pca = train_pca.drop("features_normalized").withColumnRenamed("pca_features", "features")
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

start_time = time.time()
test_pca = test_pca.drop("features_normalized").withColumnRenamed("pca_features", "features")
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Verify the changes
#train_pca.show()
#test_pca.show()

Execution time: 0.011535406112670898 seconds
Execution time: 0.00988459587097168 seconds


In [16]:
# Create the SVM model
start_time = time.time()
svm = LinearSVC(labelCol="label_tactic_indexed", featuresCol="features", maxIter=10, regParam=0.0, tol=.00001, fitIntercept=True)

# One Vs. Rest
ovr = OneVsRest(classifier=svm, labelCol='label_tactic_indexed')

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



# Fit the model
start_time = time.time()

svm_model = ovr.fit(train_pca)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.014410972595214844 seconds


24/04/08 21:10:24 WARN DAGScheduler: Broadcasting large task binary with size 265.9 MiB
24/04/08 21:11:06 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:11:51 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:12:11 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:12:26 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:12:49 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:13:05 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:13:22 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:13:39 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:13:55 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:14:12 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 21:14:29 WARN DAGSchedu

Execution time: 4920.079784154892 seconds


In [17]:
# Make predictions
start_time = time.time()

predictions = svm_model.transform(test_pca)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.6566295623779297 seconds


In [18]:
# Evaluate the model
# Calculate accuracy
start_time = time.time()
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)
print("Accuracy:", accuracy)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate precision
start_time = time.time()
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)
print("Precision:", precision)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate recall
start_time = time.time()
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)
print("Recall:", recall)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate F1-score
start_time = time.time()
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)
print("F1-Score:", f1_score)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)

24/04/08 22:32:45 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


Accuracy: 1.0
Execution time: 93.2108747959137 seconds


24/04/08 22:34:21 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


Precision: 1.0
Execution time: 106.92829251289368 seconds


24/04/08 22:36:09 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


Recall: 1.0
Execution time: 94.78175139427185 seconds


24/04/08 22:37:40 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


F1-Score: 1.0
Execution time: 109.78808188438416 seconds
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-Score: 1.0


In [19]:
from pyspark.sql.functions import expr

start_time = time.time()

# Extract Predictions and True Labels
predictions_and_labels = predictions.select("prediction", "label_tactic_indexed")

# Calculate False Positives
false_positives = predictions_and_labels.filter((predictions_and_labels.prediction == 1) & (predictions_and_labels.label_tactic_indexed == 0)).count()

# Calculate True Negatives
true_negatives = predictions_and_labels.filter((predictions_and_labels.prediction == 0) & (predictions_and_labels.label_tactic_indexed == 0)).count()

# Calculate False Positive Rate (FPR)
fpr = false_positives / (false_positives + true_negatives)

print("False Positive Rate:", fpr)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


24/04/08 22:39:29 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/08 22:41:27 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


False Positive Rate: 0.0
Execution time: 219.57944774627686 seconds


24/04/09 00:24:55 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 12059785 ms exceeds timeout 12000000 ms
24/04/09 00:24:55 WARN SparkContext: Killing executors is not supported by current scheduler.
24/04/09 03:35:36 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoi

In [ ]:
spark.sparkContext.stop()